# 名詞限定クラスタ0：ランダム30発言の内容確認

2016〜2026年の名詞限定クラスタリング結果から、クラスタ0を固定シードで30件抽出します。発言本文を省略せず表示し、ノイズやトピックの一貫性を目視確認します。

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
INPUT_PATH = ROOT / 'analysis/clustering_2016_2026_nouns/speech_clusters.csv'
OUTPUT_PATH = ROOT / 'analysis/clustering_2016_2026_nouns/cluster_0_random_30.csv'
CLUSTER_ID = 0
SAMPLE_SIZE = 30
RANDOM_STATE = 42

## 1. クラスタ0の母集団を確認

In [2]:
df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig', parse_dates=['date'])
required = {'speechID', 'date', 'year', 'speaker', 'speakerPosition', 'speech', 'speech_cleaned', 'review_flags', 'speechURL', 'cluster', 'cluster_label', 'centroid_distance', 'representativeness'}
missing = required - set(df.columns)
assert not missing, f'必要な列がありません: {sorted(missing)}'
assert df['speechID'].is_unique
cluster_population = df.loc[df['cluster'].eq(CLUSTER_ID)].copy()
assert len(cluster_population) >= SAMPLE_SIZE
display(pd.Series({
    '全クラスタリング発言': len(df),
    'クラスタ0発言': len(cluster_population),
    'クラスタ0ラベル': cluster_population['cluster_label'].iat[0],
    '抽出件数': SAMPLE_SIZE,
    '乱数シード': RANDOM_STATE,
}).to_frame('value'))
display(cluster_population.groupby('year', as_index=False).agg(speeches=('speechID', 'size'), speakers=('speaker', 'nunique')))

,value
全クラスタリング発言,67882
クラスタ0発言,23454
クラスタ0ラベル,先程・答え・報告
抽出件数,30
乱数シード,42


,year,speeches,speakers
0,2016,1791,144
1,2017,3320,167
2,2018,3598,173
3,2019,2925,154
4,2020,3747,152
5,2021,2099,179
6,2022,1508,140
7,2023,1449,154
8,2024,1195,152
9,2025,1312,209


## 2. 固定シードで30件を抽出

In [3]:
sample = cluster_population.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
sample.insert(0, 'sample_order', range(1, SAMPLE_SIZE + 1))
sample.loc[:, 'review_decision'] = ''
sample.loc[:, 'review_notes'] = ''
assert len(sample) == SAMPLE_SIZE
assert sample['speechID'].is_unique
assert sample['cluster'].eq(CLUSTER_ID).all()
display(sample[['sample_order', 'date', 'year', 'speaker', 'speakerPosition', 'review_flags', 'centroid_distance', 'speechID']])

,sample_order,date,year,speaker,speakerPosition,review_flags,centroid_distance,speechID
64396,1,2025-01-31,2025,石破茂,内閣総理大臣,NaN,0.991035,121705261X00220250131_272
6728,2,2017-07-24,2017,安倍晋三,内閣総理大臣,NaN,0.970525,119305261X02120170724_293
14574,3,2018-02-26,2018,玉木雄一郎,NaN,NaN,0.925076,119605261X01720180226_202
3210,4,2016-02-15,2016,高橋千鶴子,NaN,NaN,0.902311,119005261X01220160215_309
27302,5,2020-04-28,2020,大串博志,NaN,NaN,0.918215,120105261X02020200428_215
12817,6,2018-11-01,2018,柴山昌彦,文部科学大臣,NaN,0.934219,119705261X00220181101_296
37438,7,2021-02-10,2021,山尾志桜里,NaN,very_short_review,0.991528,120405261X00820210210_459
7091,8,2017-03-23,2017,籠池康博,NaN,NaN,0.989698,119305261X01820170323_147
37953,9,2021-02-05,2021,秋本芳徳,総務省情報流通行政局長,very_short_review,0.995226,120405261X00520210205_042
32958,10,2020-01-27,2020,並木稔,カジノ管理委員会事務局次長,NaN,0.963482,120105261X00220200127_225


## 3. 発言内容を目視確認

`speech_cleaned` はクラスタリング対象本文、`speech` は確認用の原文です。

In [4]:
for row in sample.itertuples(index=False):
    position = row.speakerPosition if pd.notna(row.speakerPosition) and str(row.speakerPosition).strip() else '役職なし'
    flags = row.review_flags if pd.notna(row.review_flags) and str(row.review_flags).strip() else 'なし'
    cleaned = str(row.speech_cleaned).replace('\n', '  \n')
    original = str(row.speech).replace('\n', '  \n')
    display(Markdown(
        f"### {row.sample_order}. {row.speaker}（{position}）— {row.date:%Y-%m-%d}\n"
        f"- 発言ID: `{row.speechID}`\n"
        f"- レビューフラグ: {flags}\n"
        f"- 重心距離: {row.centroid_distance:.6f}\n"
        f"- [国会会議録の発言ページ]({row.speechURL})\n\n"
        f"**クリーニング本文**\n\n{cleaned}\n\n"
        f"<details><summary>原文を表示</summary>\n\n{original}\n\n</details>"
    ))

### 1. 石破茂（内閣総理大臣）— 2025-01-31
- 発言ID: `121705261X00220250131_272`
- レビューフラグ: なし
- 重心距離: 0.991035
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/121705261X00220250131/272)

**クリーニング本文**

そのようにお使いになることは、それぞれの自治体の御判断でございます。それは、ですから、地産地消というんでしょうか、そこにおける食材、これを用いる、それでこのお金を活用する、それはその自治体の御判断でございます。

<details><summary>原文を表示</summary>

○石破内閣総理大臣　そのようにお使いになることは、それぞれの自治体の御判断でございます。それは、ですから、地産地消というんでしょうか、そこにおける食材、これを用いる、それでこのお金を活用する、それはその自治体の御判断でございます。

</details>

### 2. 安倍晋三（内閣総理大臣）— 2017-07-24
- 発言ID: `119305261X02120170724_293`
- レビューフラグ: なし
- 重心距離: 0.970525
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/119305261X02120170724/293)

**クリーニング本文**

私は、基本的に、日報がもしあれば直ちに出すべきだという考え方をずっと申し上げてきたわけであります。戦闘という言葉について国会でも議論になりましたが、定義を決めている戦闘行為とは違う意味で、一般的、いわば国語辞典的な意味での戦闘という言葉を使う、これはあり得るということは私はここで答弁をしているわけでありますから、そうしたことを、出せるものは出していくというのが私の基本的な考えでありますから、もしその日報がどこかに保管されていようと、それは絶対隠蔽などしてはならないというのが私の基本的な立場であります。 それを私が指示するということは、これはあり得ないわけでございまして、このことについて私が、そういう報告を受けたら、陸自に日報があったという報告を受けたら、あるいは、それを外に出さない、こんな指示をするはずがないわけでありますし、改めて申し上げますが、黒江次官から陸自にこの日報が残っていたという報告を受けたことは全くありません。

<details><summary>原文を表示</summary>

○安倍内閣総理大臣　私は、基本的に、日報がもしあれば直ちに出すべきだという考え方をずっと申し上げてきたわけであります。戦闘という言葉について国会でも議論になりましたが、定義を決めている戦闘行為とは違う意味で、一般的、いわば国語辞典的な意味での戦闘という言葉を使う、これはあり得るということは私はここで答弁をしているわけでありますから、そうしたことを、出せるものは出していくというのが私の基本的な考えでありますから、もしその日報がどこかに保管されていようと、それは絶対隠蔽などしてはならないというのが私の基本的な立場であります。  
　それを私が指示するということは、これはあり得ないわけでございまして、このことについて私が、そういう報告を受けたら、陸自に日報があったという報告を受けたら、あるいは、それを外に出さない、こんな指示をするはずがないわけでありますし、改めて申し上げますが、黒江次官から陸自にこの日報が残っていたという報告を受けたことは全くありません。

</details>

### 3. 玉木雄一郎（役職なし）— 2018-02-26
- 発言ID: `119605261X01720180226_202`
- レビューフラグ: なし
- 重心距離: 0.925076
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/119605261X01720180226/202)

**クリーニング本文**

いや、不適切なことは明確に会計検査院は指摘しているんですよ。でも、誰も責任者がいない。恐ろしい話じゃないかと思いませんか、大臣、これは。 では、わかりました。行政の中にそういった不適切な人がいないんですね。いないからこそ、では、それ以外の要素でこの不適切さが導かれたんではないかと言わざるを得ない。 だから、この委員会でも何度も議論のあった、総理、あるいは総理の奥様の安倍昭恵さんが名誉校長を務めていた、そういったことが何らかの形でこの不適切さの根源になっているんではないかということが疑われているんですね。 今、もし誰か行政の中で不適切なことをした人がいれば、その人の責任を問えばいいでしょう。でも、それはいないんですよね。だったら、それ以外のところに責任を求めざるを得なくなっているのが、この問題が長引いている理由の一つなんではないですか。 総理に伺いますけれども、私、前回の質問のときにも少し冒頭言いましたが、大阪教育庁に行ったら、確かに申請のところには開成小学校とか瑞穂の国記念小学院と書いてあるんですが、その前の相談の段階では、申請のとき、初めは仮称として安倍晋三記念小学院だったと担当の課長が言っていましたよ。だから、やはりこういうことが、今、麻生大臣が答えられなかった、行政以外のところで誰か不適切さの理由、根源があるとしたら、こういうところにやはりあると言わざるを得ないんじゃないですかね。 総理に伺います。 実は、証人喚問でも籠池氏がこういうふうにおっしゃっていますね。「安倍昭恵先生の方から御主人であらっしゃる安倍首相の方に、どうですか、安倍晋三記念小学院という名前はどうですかということを二度、三度とお聞きいただいて、当初はオーケーということでした」とあります。 もしこういうのがうそだったら、偽証罪で訴追すればいいと思うんですよ。それをしないということは、認めたということじゃないんですか。いかがですか、総理。

<details><summary>原文を表示</summary>

○玉木委員　いや、不適切なことは明確に会計検査院は指摘しているんですよ。でも、誰も責任者がいない。恐ろしい話じゃないかと思いませんか、大臣、これは。  
　では、わかりました。行政の中にそういった不適切な人がいないんですね。いないからこそ、では、それ以外の要素でこの不適切さが導かれたんではないかと言わざるを得ない。  
　だから、この委員会でも何度も議論のあった、総理、あるいは総理の奥様の安倍昭恵さんが名誉校長を務めていた、そういったことが何らかの形でこの不適切さの根源になっているんではないかということが疑われているんですね。  
　今、もし誰か行政の中で不適切なことをした人がいれば、その人の責任を問えばいいでしょう。でも、それはいないんですよね。だったら、それ以外のところに責任を求めざるを得なくなっているのが、この問題が長引いている理由の一つなんではないですか。  
　総理に伺いますけれども、私、前回の質問のときにも少し冒頭言いましたが、大阪教育庁に行ったら、確かに申請のところには開成小学校とか瑞穂の国記念小学院と書いてあるんですが、その前の相談の段階では、申請のとき、初めは仮称として安倍晋三記念小学院だったと担当の課長が言っていましたよ。だから、やはりこういうことが、今、麻生大臣が答えられなかった、行政以外のところで誰か不適切さの理由、根源があるとしたら、こういうところにやはりあると言わざるを得ないんじゃないですかね。  
　総理に伺います。  
　実は、証人喚問でも籠池氏がこういうふうにおっしゃっていますね。「安倍昭恵先生の方から御主人であらっしゃる安倍首相の方に、どうですか、安倍晋三記念小学院という名前はどうですかということを二度、三度とお聞きいただいて、当初はオーケーということでした」とあります。  
　もしこういうのがうそだったら、偽証罪で訴追すればいいと思うんですよ。それをしないということは、認めたということじゃないんですか。いかがですか、総理。

</details>

### 4. 高橋千鶴子（役職なし）— 2016-02-15
- 発言ID: `119005261X01220160215_309`
- レビューフラグ: なし
- 重心距離: 0.902311
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/119005261X01220160215/309)

**クリーニング本文**

ありがとうございます。 一定の類型化をしたけれども、個別事情を見て対応するべきだという答弁だと思っております。 それで、もう一度東電に伺いたいと思うんです。 先ほど紹介した「三つの誓い」を決めた新・総合特別事業計画の中では、「東電と被害者の方々との間に認識の齟齬がある場合であっても解決に向けて真摯に対応するよう、ＡＤＲの和解案を尊重する。」とまで言っているんですね。 だけれども、今の東電の対応は、さっき読み上げられたように、客観的事実からすると原発事故との相当因果関係が明らかに認めがたい請求については、その内容を十分に吟味、検討した上で慎重に対応する必要があると言っているんです。損害がある以上、当然、賠償していくという考えに変わりはないと、これまでも東電はお答えになっております。しかし、その前提に、相当因果関係があり、合理的な範囲内ではあるがと言っている。 おかしくないですか。つまり、加害者である東電が賠償するべき根拠、相当因果関係があるかないかをジャッジする、これはおかしくないですか。逆立ちしていませんか。

<details><summary>原文を表示</summary>

○高橋（千）委員　ありがとうございます。  
　一定の類型化をしたけれども、個別事情を見て対応するべきだという答弁だと思っております。  
　それで、もう一度東電に伺いたいと思うんです。  
　先ほど紹介した「三つの誓い」を決めた新・総合特別事業計画の中では、「東電と被害者の方々との間に認識の齟齬がある場合であっても解決に向けて真摯に対応するよう、ＡＤＲの和解案を尊重する。」とまで言っているんですね。  
　だけれども、今の東電の対応は、さっき読み上げられたように、客観的事実からすると原発事故との相当因果関係が明らかに認めがたい請求については、その内容を十分に吟味、検討した上で慎重に対応する必要があると言っているんです。損害がある以上、当然、賠償していくという考えに変わりはないと、これまでも東電はお答えになっております。しかし、その前提に、相当因果関係があり、合理的な範囲内ではあるがと言っている。  
　おかしくないですか。つまり、加害者である東電が賠償するべき根拠、相当因果関係があるかないかをジャッジする、これはおかしくないですか。逆立ちしていませんか。

</details>

### 5. 大串博志（役職なし）— 2020-04-28
- 発言ID: `120105261X02020200428_215`
- レビューフラグ: なし
- 重心距離: 0.918215
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/120105261X02020200428/215)

**クリーニング本文**

ちょっと今になって新しいことがいろいろ言われて、極めて不透明な感じがするんですけれども、先ほどユースビオは布の調達及びその調整に関する業務みたいなことをやっていたというふうにおっしゃいました。よって契約の対象なんだみたいなことをおっしゃいましたけれども、それにしても、三月にユースビオが定款に会社の目的として書いていた中には、布の調達とかそういったもののコーディネーションというのはないんですよ。そういった会社になぜ布マスクの調達の業務が。しかも、随意契約です。 随意契約というのは、この会社にやってくれというふうに政府の方からお願いして、急ぐから、あなたしかいないんだということなんです。そのほかの会社は、伊藤忠さん、興和さん、そしてマツオカコーポレーションさん、大きな会社です。代表的な会社です。だから、政府がそこに目をつけてお願いというのはわかります。実際、伊藤忠さんはプレスリリースの中でしっかりそこの辺は言われていて、政府が調達できなかったから、自分たちはこれを何とかしなきゃいかぬと思って協力したということは書かれていらっしゃいます。 この伊藤忠さんやあるいは興和さんと比べて、ちょっと社の規模としては、これは資本金一千万だということなので、かなり違うここが、どういう経緯で、大臣、マスクを政府側から随契で早く納めてくれというふうに言うような対象にどういう経緯でここはなったんですか。

<details><summary>原文を表示</summary>

○大串（博）委員　ちょっと今になって新しいことがいろいろ言われて、極めて不透明な感じがするんですけれども、先ほどユースビオは布の調達及びその調整に関する業務みたいなことをやっていたというふうにおっしゃいました。よって契約の対象なんだみたいなことをおっしゃいましたけれども、それにしても、三月にユースビオが定款に会社の目的として書いていた中には、布の調達とかそういったもののコーディネーションというのはないんですよ。そういった会社になぜ布マスクの調達の業務が。しかも、随意契約です。  
　随意契約というのは、この会社にやってくれというふうに政府の方からお願いして、急ぐから、あなたしかいないんだということなんです。そのほかの会社は、伊藤忠さん、興和さん、そしてマツオカコーポレーションさん、大きな会社です。代表的な会社です。だから、政府がそこに目をつけてお願いというのはわかります。実際、伊藤忠さんはプレスリリースの中でしっかりそこの辺は言われていて、政府が調達できなかったから、自分たちはこれを何とかしなきゃいかぬと思って協力したということは書かれていらっしゃいます。  
　この伊藤忠さんやあるいは興和さんと比べて、ちょっと社の規模としては、これは資本金一千万だということなので、かなり違うここが、どういう経緯で、大臣、マスクを政府側から随契で早く納めてくれというふうに言うような対象にどういう経緯でここはなったんですか。

</details>

...

### 26. 安倍晋三（内閣総理大臣）— 2020-02-03
- 発言ID: `120105261X00520200203_263`
- レビューフラグ: なし
- 重心距離: 0.991915
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/120105261X00520200203/263)

**クリーニング本文**

政府として、今回の対応、政府一丸となって全力を尽くしているところでございますが、当然、その対応において発生した公文書については、法令に従ってしっかりと適切に残していくということになると思います。

<details><summary>原文を表示</summary>

○安倍内閣総理大臣　政府として、今回の対応、政府一丸となって全力を尽くしているところでございますが、当然、その対応において発生した公文書については、法令に従ってしっかりと適切に残していくということになると思います。

</details>

### 27. 階猛（役職なし）— 2017-02-02
- 発言ID: `119305261X00520170202_037`
- レビューフラグ: なし
- 重心距離: 0.988566
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/119305261X00520170202/37)

**クリーニング本文**

具体的な対象犯罪を聞いているのではなくて、対象犯罪を懲役何年以上とかいうことで広く設けた上で、それについての準備罪を設けるのか、それとも特定の犯罪について準備罪を設けるのか、共謀罪を設けるのか、どちらをとるのかということを聞いているんです。方向性を聞いているんです。

<details><summary>原文を表示</summary>

○階委員　具体的な対象犯罪を聞いているのではなくて、対象犯罪を懲役何年以上とかいうことで広く設けた上で、それについての準備罪を設けるのか、それとも特定の犯罪について準備罪を設けるのか、共謀罪を設けるのか、どちらをとるのかということを聞いているんです。方向性を聞いているんです。

</details>

### 28. 山下貴司（法務大臣）— 2019-02-15
- 発言ID: `119805261X00820190215_252`
- レビューフラグ: very_short_review
- 重心距離: 0.994553
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/119805261X00820190215/252)

**クリーニング本文**

冷静でございます。失礼いたしました。

<details><summary>原文を表示</summary>

○山下国務大臣　冷静でございます。失礼いたしました。

</details>

### 29. 菅義偉（内閣総理大臣）— 2020-11-04
- 発言ID: `120305261X00320201104_340`
- レビューフラグ: なし
- 重心距離: 0.966732
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/120305261X00320201104/340)

**クリーニング本文**

まず、八月三十一日に学術会議から総理大臣宛てに百五名の推薦名簿が提出され、担当の内閣府からその内容の報告を受けて、私が官房長官時代から持っていた懸念というものを伝えました。こうした懸念点や任命の考え方を官房長官を通じて内閣府に伝えて、九月二十四日に内閣府が九十九名を任命する旨の決裁を起案して、九月二十八日ですね、決裁を行ったのは。

<details><summary>原文を表示</summary>

○菅内閣総理大臣　まず、八月三十一日に学術会議から総理大臣宛てに百五名の推薦名簿が提出され、担当の内閣府からその内容の報告を受けて、私が官房長官時代から持っていた懸念というものを伝えました。こうした懸念点や任命の考え方を官房長官を通じて内閣府に伝えて、九月二十四日に内閣府が九十九名を任命する旨の決裁を起案して、九月二十八日ですね、決裁を行ったのは。

</details>

### 30. 小川淳也（役職なし）— 2020-01-28
- 発言ID: `120105261X00320200128_080`
- レビューフラグ: なし
- 重心距離: 0.975520
- [国会会議録の発言ページ](https://kokkai.ndl.go.jp/txt/120105261X00320200128/80)

**クリーニング本文**

それは、後援会名簿にも記録が残っていない、たどりようはないという御答弁ですか。

<details><summary>原文を表示</summary>

○小川委員　それは、後援会名簿にも記録が残っていない、たどりようはないという御答弁ですか。

</details>

## 4. レビュー用CSVを保存

`review_decision` と `review_notes` は人手確認用の空欄です。Excelでも開けるUTF-8 BOM付きで保存します。

In [5]:
export_columns = [
    'sample_order', 'speechID', 'date', 'year', 'speaker', 'speakerPosition',
    'speech', 'speech_cleaned', 'review_flags', 'speechURL', 'cluster',
    'cluster_label', 'centroid_distance', 'representativeness',
    'review_decision', 'review_notes',
]
sample.loc[:, export_columns].to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
saved = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
assert len(saved) == SAMPLE_SIZE
assert saved['speechID'].tolist() == sample['speechID'].tolist()
print(f'保存完了: {OUTPUT_PATH}')

保存完了: /Users/tasskkiu2/kokkai/analysis/clustering_2016_2026_nouns/cluster_0_random_30.csv
